# Producing Messages to Kafka Cluster

Producing messages simply means sending files in DE, actual feeding the system with data. In real life scenarios, especially when dealing with large data volumes, the amount sent is in PB or TB.
Below is an illustration of how to send data using Python to our Kafka cluster

### Send single messages to our Kafka cluster

In [2]:
# Create a dictionary for your configuration
conf = {
    'bootstrap.servers': 'pkc-921jm.us-east-2.aws.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': 'GARCXDH6ULX5A2VJ',
    'sasl.password': 'cfltmwb0lU9jqFNOhZd1einMY0YJ/nRqHoW4TjcErUYFl7bEyqT+vA/eJC+E0Ouw',
    'client.id': 'my-python-app'
}

# Example: How you would use it with the Kafka Producer
# from confluent_kafka import Producer
# p = Producer(conf)


In [3]:
!pip install confluent_kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 46.2 MB/s eta 0:00:00


In [4]:
from confluent_kafka import Producer
import json
import time

In [6]:
import pandas as pd
import json

In [7]:
df = pd.read_csv('/content/first_100_customers.csv')

In [8]:
df.head()

,customer_id,name,city,state,country,registration_date,is_active
0,0,Customer_0,Pune,Maharashtra,India,2023-06-29,False
1,1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True
2,2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True
3,3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False
4,4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False


In [9]:
json_records = df.to_dict(orient='records')
json_file = 'customers.json'
with open(json_file,'w') as file:
  json.dump(json_records, file, indent=4)

print('File converted to JSON. Well done Mercy')

File converted to JSON. Well done Mercy


In [10]:
producer = Producer(conf)

In [16]:
topic ='ecommerce'
with open ('customers.json','r') as file:
  customers_data = json.load(file)

value = customers_data[0]
key = value['customer_id']

print(key,value)

0 {'customer_id': 0, 'name': 'Customer_0', 'city': 'Pune', 'state': 'Maharashtra', 'country': 'India', 'registration_date': '2023-06-29', 'is_active': False}


In [17]:
producer.produce(topic, key=str(key).encode('utf-8'), value=str(value).encode('utf-8'))

### Send multiple messages to our Kafka cluster

In [27]:
def delivery_status(err,msg):
  if(err):
    print(f"Message delivery failed: {err}")
  else:
    print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

for record in customers_data:
  try:

    message_value = json.dumps(record)
    message_key = str(int(record['customer_id']) + 1000).encode('utf-8')

    producer.produce(topic,key = message_key,value=message_value,callback = delivery_status)
    producer.poll(1)

  except Exception as e:
    print(f"Error sending messages: {e}")

producer.flush()

print("Message send to kafka successfully. Well done M!")

Message delivered to ecommerce [0] at offset 45
Message delivered to ecommerce [2] at offset 35
Message delivered to ecommerce [0] at offset 46
Message delivered to ecommerce [2] at offset 36
Message delivered to ecommerce [1] at offset 22
Message delivered to ecommerce [2] at offset 37
Message delivered to ecommerce [0] at offset 47
Message delivered to ecommerce [1] at offset 23
Message delivered to ecommerce [2] at offset 38
Message delivered to ecommerce [2] at offset 39
Message delivered to ecommerce [0] at offset 48
Message delivered to ecommerce [1] at offset 24
Message delivered to ecommerce [2] at offset 40
Message delivered to ecommerce [2] at offset 41
Message delivered to ecommerce [2] at offset 42
Message delivered to ecommerce [2] at offset 43
Message delivered to ecommerce [2] at offset 44
Message delivered to ecommerce [1] at offset 25
Message delivered to ecommerce [1] at offset 26
Message delivered to ecommerce [0] at offset 49
Message delivered to ecommerce [1] at of

# Consuming Messages on Kafka

In DE, consuming is like reading mails that have been delivered to you.

In [19]:
!pip install confluent_kafka

In [20]:
from confluent_kafka import Consumer, KafkaError, KafkaException
import json
import time

In [22]:
# Create a dictionary for your configuration
conf = {
    'bootstrap.servers': 'pkc-921jm.us-east-2.aws.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': 'GARCXDH6ULX5A2VJ',
    'sasl.password': 'cfltmwb0lU9jqFNOhZd1einMY0YJ/nRqHoW4TjcErUYFl7bEyqT+vA/eJC+E0Ouw',
    'client.id': 'my-python-app',
    'auto.offset.reset':'earliest',
    'group.id':'customer_group'
}

# Example: How you would use it with the Kafka Producer
# from confluent_kafka import Producer
# p = Producer(conf)


In [24]:
consumer = Consumer(conf)

In [25]:
topic = 'ecommerce'
consumer.subscribe([topic])


def process_message(message):
  try:
    if message.error():
      if(message.error.code()) == KafkaError._PARTITION_EOF:
        print('End of partition reached {0}/{1}')
      else:
        raise KafkaException(message.error())
    else:
      key = message.key().decode('utf-8')
      value = json.loads(message.value().decode('utf-8'))

      print(f"Recieved message : Key {key} , Value : {value}")
  except Exception as e:
    print(f"Error consuming/processing message : {e}")



In [26]:

# Poll messages Continously
try :
  print(" Listening for messages, press Ctrl + C to exit")

  while True:
    message = consumer.poll(timeout=1.0)
    if(message):
      process_message(message)

except KeyboardInterrupt:
  print("Interrupted by user, shutting down consumer")

finally:
  consumer.close()

 Listening for messages, press Ctrl + C to exit
Recieved message : Key 123 , Value : {}
Error consuming/processing message : Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Recieved message : Key 1001 , Value : {'customer_id': 1, 'name': 'Customer_1', 'city': 'Bangalore', 'state': 'Tamil Nadu', 'country': 'India', 'registration_date': '2023-12-07', 'is_active': True}
Recieved message : Key 1003 , Value : {'customer_id': 3, 'name': 'Customer_3', 'city': 'Bangalore', 'state': 'Karnataka', 'country': 'India', 'registration_date': '2023-10-17', 'is_active': False}
Recieved message : Key 1005 , Value : {'customer_id': 5, 'name': 'Customer_5', 'city': 'Hyderabad', 'state': 'Karnataka', 'country': 'India', 'registration_date': '2023-07-28', 'is_active': False}
Recieved message : Key 1008 , Value : {'customer_id': 8, 'name': 'Customer_8', 'city': 'Pune', 'state': 'Karnataka', 'country': 'India', 'registration_date': '2023-06-22', 'is_active': True}
Recieved message 